<a href="https://colab.research.google.com/github/neowork77/BU-Dorms-RAG-CHAT/blob/main/Web_Scraping_Dorms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# =========================================================
# INSTALL
# =========================================================

!pip install requests beautifulsoup4 lxml tqdm -q

In [ ]:
# =========================================================
# IMPORT
# =========================================================

import requests
import pandas as pd
from tqdm import tqdm
from bs4 import BeautifulSoup
import json
import re
import random
import time

# 📍 **config**

In [ ]:
BASE_URL = "https://propertyhub.in.th"

START_URL = "https://propertyhub.in.th/เช่าคอนโด/ม-กรุงเทพ-ศูนย์รังสิต"

TOTAL_PAGES = 16

HEADERS = {
    "User-Agent": (
        "Mozilla/5.0 (Macintosh; Intel Mac OS X 10_15_7) "
        "AppleWebKit/537.36 (KHTML, like Gecko) "
        "Chrome/147.0.0.0 Safari/537.36"
    )
}

# 📍 **helper**

In [ ]:
def clean_text(text):

    if text is None:
        return ""

    text = str(text)

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# ---------------------------------------------------------

def find_next_data(html):

    pattern = r'<script id="__NEXT_DATA__" type="application/json">(.*?)</script>'

    match = re.search(pattern, html, re.DOTALL)

    if not match:
        return None

    try:
        return json.loads(match.group(1))

    except:
        return None

# ---------------------------------------------------------

def recursive_find(obj, target_key):

    results = []

    if isinstance(obj, dict):

        for k, v in obj.items():

            if k == target_key:
                results.append(v)

            results.extend(
                recursive_find(v, target_key)
            )

    elif isinstance(obj, list):

        for item in obj:

            results.extend(
                recursive_find(item, target_key)
            )

    return results

# 🌟 **STEP 1 : GET ALL LINKS**

In [ ]:
listing_links = []

for page in range(1, TOTAL_PAGES + 1):

    if page == 1:
        url = START_URL
    else:
        url = f"{START_URL}/{page}"

    print(f"\nPAGE {page}")

    response = requests.get(
        url,
        headers=HEADERS
    )

    html = response.text

    # -----------------------------------------------------
    # ดึง link ด้วย regex
    # -----------------------------------------------------

    links = re.findall(
        r'href="(/listings/[^"]+)"',
        html
    )

    links = list(set(links))

    for link in links:

        full_url = BASE_URL + link

        if full_url not in listing_links:
            listing_links.append(full_url)

    print("FOUND:", len(links))

    time.sleep(0.2)

print("\nTOTAL LINKS =", len(listing_links))


PAGE 1
FOUND: 60

PAGE 2
FOUND: 64

PAGE 3
FOUND: 64

PAGE 4
FOUND: 64

PAGE 5
FOUND: 64

PAGE 6
FOUND: 64

PAGE 7
FOUND: 64

PAGE 8
FOUND: 64

PAGE 9
FOUND: 64

PAGE 10
FOUND: 64

PAGE 11
FOUND: 64

PAGE 12
FOUND: 64

PAGE 13
FOUND: 64

PAGE 14
FOUND: 64

PAGE 15
FOUND: 64

PAGE 16
FOUND: 52

TOTAL LINKS = 948


# 🌟 **STEP 2 : SCRAPE DETAIL**

In [ ]:
from concurrent.futures import ThreadPoolExecutor, as_completed

all_data = []

def scrape_detail(link):

    try:

        response = requests.get(
            link,
            headers=HEADERS,
            timeout=10
        )

        html = response.text

        json_data = find_next_data(html)

        if not json_data:
            return None

        # =================================================
        # PROJECT
        # =================================================

        project_name = ""

        projects = recursive_find(json_data, "project")

        if projects:

            project = projects[0]

            if isinstance(project, dict):

                project_name = clean_text(
                    project.get("name", "")
                )

        # =================================================
        # LOCATION
        # =================================================

        latitude = ""
        longitude = ""

        if projects:

            project = projects[0]

            location = project.get("location", {})

            latitude = location.get("lat", "")
            longitude = location.get("lng", "")

        # =================================================
        # PRICE
        # =================================================

        price = ""

        monthly_prices = recursive_find(
            json_data,
            "monthly"
        )

        if monthly_prices:

            monthly = monthly_prices[0]

            if isinstance(monthly, dict):

                price = monthly.get("price", "")

        # =================================================
        # ROOM INFO
        # =================================================

        room_type = ""
        room_size = ""
        floor = ""
        building = ""

        room_types = recursive_find(json_data, "roomType")
        room_areas = recursive_find(json_data, "roomArea")
        floors = recursive_find(json_data, "onFloor")
        buildings = recursive_find(json_data, "building")

        if room_types:
            room_type = room_types[0]

        if room_areas:
            room_size = room_areas[0]

        if floors:
            floor = floors[0]

        if buildings:
            building = buildings[0]

        # =================================================
        # HTML PARSE
        # =================================================

        soup = BeautifulSoup(html, "lxml")

        # =================================================
        # ROOM FACILITIES
        # =================================================

        room_facilities = []

        room_section = soup.select_one(
            "div.sc-9do0cc-3"
        )

        if room_section:

            items = room_section.select(
                "div.sc-9do0cc-0.hKmUMz"
            )

            for item in items:

                span = item.find("span")

                if span:

                    text = clean_text(span.text)

                    if text:
                        room_facilities.append(text)

        # =================================================
        # PROJECT FACILITIES
        # =================================================

        project_facilities = []

        project_section = soup.select_one(
            "div.sc-9do0cc-4"
        )

        if project_section:

            items = project_section.select(
                "div.sc-9do0cc-0.hKmUMz"
            )

            for item in items:

                span = item.find("span")

                if span:

                    text = clean_text(span.text)

                    if text:
                        project_facilities.append(text)

        # =================================================
        # IMAGES
        # =================================================

        image_tags = soup.select(
            "img.sc-1wp7ebx-5"
        )

        image_urls = []

        for img in image_tags:

            src = img.get("src")

            if src and src not in image_urls:
                image_urls.append(src)

        image_urls = image_urls[:4]

        # =================================================
        # RETURN
        # =================================================

        return {

            "project_name": project_name,

            "url": link,

            "price": price,

            "room_type": room_type,

            "room_size_sqm": room_size,

            "floor": floor,

            "building": building,

            "latitude": latitude,

            "longitude": longitude,

            "room_facilities":
                " , ".join(room_facilities),

            "project_facilities":
                " , ".join(project_facilities),

            "image_1":
                image_urls[0]
                if len(image_urls) > 0 else "",

            "image_2":
                image_urls[1]
                if len(image_urls) > 1 else "",

            "image_3":
                image_urls[2]
                if len(image_urls) > 2 else "",

            "image_4":
                image_urls[3]
                if len(image_urls) > 3 else "",
        }

    except Exception as e:

        print("ERROR:", link)
        print(e)

        return None


# =================================================
# PARALLEL RUN
# =================================================

with ThreadPoolExecutor(max_workers=80) as executor:

    futures = [
        executor.submit(scrape_detail, link)
        for link in listing_links
    ]

    for future in tqdm(
        as_completed(futures),
        total=len(futures)
    ):

        result = future.result()

        if result:
            all_data.append(result)

  1%|▏         | 14/948 [00:10<18:03,  1.16s/it]

ERROR:ERROR: https://propertyhub.in.th/listings/เช่าคอนโด-แอททิจูด-บียู-คลองหนึ่ง-คลองหลวง-ปทุมธานี-cx-104388-ทักไลน์-connexproperty-ตอบทันที-ทีมงานมืออาชีพ--ad030950---4680977
HTTPSConnectionPool(host='propertyhub.in.th', port=443): Read timed out. (read timeout=10)
 https://propertyhub.in.th/listings/เช่าคอนโด-เคฟ-ทียู-คลองหนึ่ง-คลองหลวง-ปทุมธานี-cx-97107-ทักไลน์-connexproperty-ตอบทันที-ทีมงานมืออาชีพ--aa7909b9---4443076
HTTPSConnectionPool(host='propertyhub.in.th', port=443): Read timed out. (read timeout=10)


  2%|▏         | 16/948 [00:10<12:48,  1.21it/s]

ERROR: https://propertyhub.in.th/listings/rb24235-ให้เช่าถูกมากkave-wonderlandแอดไลน์-119dgdea-มี-ด้วย-แอดมินตอบไว---5983883
HTTPSConnectionPool(host='propertyhub.in.th', port=443): Read timed out. (read timeout=10)


 10%|█         | 97/948 [00:39<15:02,  1.06s/it]

...

 36%|███▌      | 343/948 [02:03<02:15,  4.45it/s]

ERROR: https://propertyhub.in.th/listings/ให้เช่าคอนโด-ฟ้าโดม-ชั้น-2-ตึก-a-40-ตรม-ติดสวน-เฟอร์ครบ-เครื่องใช้ไฟฟ้าครบ-ใกล้-ม-ธรรมศาสตร์รังสิต-tr0732---3342366
HTTPSConnectionPool(host='propertyhub.in.th', port=443): Read timed out. (read timeout=10)


 36%|███▋      | 346/948 [02:04<02:57,  3.39it/s]

ERROR: https://propertyhub.in.th/listings/ให้เช่าคอนโด-ฟ้าโดม-ชั้น-7-ห้อง-35-ตรม-เครื่องใช้ไฟฟ้าครบ-ใกล้-ม-ธรรมศาสตร์รังสิต-tr0104---2617692
HTTPSConnectionPool(host='propertyhub.in.th', port=443): Read timed out. (read timeout=10)


100%|██████████| 948/948 [05:22<00:00,  2.94it/s]


# 📍 **preview**

In [ ]:
df = pd.DataFrame(all_data)
df.head()

,project_name,url,price,room_type,room_size_sqm,floor,building,latitude,longitude,room_facilities,project_facilities,image_1,image_2,image_3,image_4
0,ดีคอนโด ชายน์ รังสิต,https://propertyhub.in.th/listings/26-ตร-ม-11-...,11000.0,ONE_BED_ROOM,26.00,8,None,14.062188,100.608401,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ , เครื่องทำน้ำ...","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...
1,ดีคอนโด แคมปัส รีสอร์ท รังสิต,https://propertyhub.in.th/listings/rb23922-ให้...,8000.0,STUDIO,29.00,2,A,14.063015,100.60736,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...
2,ดีคอนโด ไฮป์ รังสิต,https://propertyhub.in.th/listings/ให้เช่า-d-c...,9000.0,ONE_BED_ROOM,27.52,3,None,14.041953,100.616681,เฟอร์นิเจอร์,"ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...
3,เคฟ วันเดอร์แลนด์,https://propertyhub.in.th/listings/เช่าคอนโด-เ...,13000.0,ONE_BED_ROOM,25.00,3,None,14.065818,100.611212,,"ลิฟท์ , ที่จอดรถจักรยานยนต์ , การรักษาความปลอด...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...
4,ดีคอนโด วิวิด รังสิต,https://propertyhub.in.th/listings/ให้เช่า-dco...,11000.0,ONE_BED_ROOM,1.00,4,A,14.04167,100.616693,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ , เครื่องทำน้ำ...","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...


# 📍 **cleaning**

In [ ]:
import numpy as np
# เปลี่ยน string ว่าง -> NaN
df = df.replace("", np.nan)

# เช็ค missing แต่ยังไม่ลบ
print(df.isna().sum())

project_name            0
url                     0
price                  10
room_type               0
room_size_sqm           0
floor                   2
building              387
latitude                4
longitude               4
room_facilities        49
project_facilities      8
image_1                 8
image_2                12
image_3                14
image_4                25
dtype: int64


/tmp/ipykernel_11634/271594810.py:3: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("", np.nan)


In [ ]:
# ลบออกเนื่องจากเป็นข้อมูลที่ใช้งานไม่ได้ เพราะ ไม่ใช่ห้องพักจริง ๆ เป็น banner ที่ติดมา
df = df.dropna(
    subset=["latitude", "longitude"]
).reset_index(drop=True)

print(df.isna().sum())

project_name            0
url                     0
price                  10
room_type               0
room_size_sqm           0
floor                   1
building              383
latitude                0
longitude               0
room_facilities        45
project_facilities      4
image_1                 4
image_2                 8
image_3                10
image_4                21
dtype: int64


In [ ]:
# ดู unique project_name
df["project_name"].unique()

array(['ดีคอนโด ชายน์ รังสิต', 'ดีคอนโด แคมปัส รีสอร์ท รังสิต',
       'ดีคอนโด ไฮป์ รังสิต', 'เคฟ วันเดอร์แลนด์', 'ดีคอนโด วิวิด รังสิต',
       'เทอร์ร่า เรสซิเดนซ์', 'เคฟ ทียู', 'เคฟ คอนโด',
       'จิณณ์ เวลบีอิ้ง เคาน์ตี้', 'แอททิจูด บียู', 'ซูม คอนโด 49',
       'เคฟ เอวา', 'พลัม คอนโด พาร์ค รังสิต', 'โมดิซ ลอนช์',
       'เคฟ ทาวน์ ชิฟท์', 'พลัม รังสิต เฟรช', 'เคฟ ทาวน์ ไอส์แลนด์',
       'คอมมอน ทียู', 'เดอะ คิทท์ คลองหลวง', 'โมดิซ อาวองการ์ด',
       'พลัม คอนโด รังสิต อไลฟ์', 'บี คอนโด พหลโยธิน', 'เคฟ ทาวน์ สเปซ',
       'ดีคอนโด แคมปัส ไฮด์อเวย์', 'พลัม คอนโด รังสิต อไลฟ์ 2',
       'พลัม คอนโด ปาร์ค รังสิต เฟส2', 'ดีคอนโด แคมปัส โดม - รังสิต',
       'เคฟ ทาวน์ โคโลนี', 'เพลิน เพลิน เจน วายด์ รังสิต - คลองหลวง',
       'ฟ้าโดม', 'พลัม คอนโด ปาร์ค รังสิต เฟส 3', 'แกรนด์ โมเดิร์น คอนโด',
       'เพลิน เพลิน เจน วายด์ รังสิต - คลองหลวง 2',
       'ดีคอนโด แคมปัส รังสิต 2', 'บ้านเอื้ออาทรพหลโยธิน ซอยคุณพระ',
       'บ้านร่วมทางฝัน 3'], dtype=object)

In [ ]:
project_map = {

    # =====================================================
    # KAVE
    # =====================================================

    "เคฟ วันเดอร์แลนด์":
        "Kave Wonderland (เคฟ วันเดอร์แลนด์)",
    "เคฟ ทียู":
        "KAVE TU (เคฟ ทียู)",
    "เคฟ คอนโด":
        "Kave Condo (เคฟ คอนโด)",
    "เคฟ เอวา":
        "Kave Ava (เคฟ เอวา)",
    "เคฟ ทาวน์ ชิฟท์":
        "Kave Town Shift (เคฟ ทาวน์ ชิฟท์)",
    "เคฟ ทาวน์ ไอส์แลนด์":
        "Kave Town Island (เคฟ ทาวน์ ไอส์แลนด์)",
    "เคฟ ทาวน์ โคโลนี":
        "Kave Town Colony (เคฟ ทาวน์ โคโลนี)",
    "เคฟ ทาวน์ สเปซ":
        "KAVE Town Space (เคฟ ทาวน์ สเปซ)",

    # =====================================================
    # PLUM
    # =====================================================

    "พลัม คอนโด พาร์ค รังสิต":
        "Plum Condo Park Rangsit (พลัม คอนโด พาร์ค รังสิต)",
    "พลัม คอนโด ปาร์ค รังสิต เฟส2":
        "Plum Condo Park Rangsit Phase 2 (พลัม คอนโด ปาร์ค รังสิต เฟส2)",
    "พลัม คอนโด รังสิต อไลฟ์":
        "Plum Condo Rangsit Alive (พลัม คอนโด รังสิต อไลฟ์)",
    "พลัม คอนโด รังสิต อไลฟ์ 2":
        "PLUM CONDO RANGSIT ALIVE 2 (พลัม คอนโด รังสิต อไลฟ์ 2)",
    "พลัม รังสิต เฟรช":
        "Plum Rangsit Fresh (พลัม รังสิต เฟรช)",
    "พลัม คอนโด ปาร์ค รังสิต เฟส 3":
        "Plum Condo Park Rangsit Phase 3 (พลัม คอนโด ปาร์ค รังสิต เฟส 3)",

    # =====================================================
    # MODIZ
    # =====================================================

    "โมดิซ อาวองการ์ด":
        "Modiz Avantgarde (โมดิซ อาวองการ์ด)",
    "โมดิซ ลอนช์":
        "Modiz Launch (โมดิซ ลอนช์)",

    # =====================================================
    # DCONDO
    # =====================================================

    "ดีคอนโด ไฮป์ รังสิต":
        "dcondo hype rangsit (ดีคอนโด ไฮป์ รังสิต)",
    "ดีคอนโด วิวิด รังสิต":
        "dcondo vivid Rangsit (ดีคอนโด วิวิด รังสิต)",
    "ดีคอนโด ชายน์ รังสิต":
        "dcondo Shine Rangsit (ดีคอนโด ชายน์ รังสิต)",
    "ดีคอนโด แคมปัส รีสอร์ท รังสิต":
        "dcondo Campus Resort Rangsit (ดีคอนโด แคมปัส รีสอร์ท รังสิต)",
    "ดีคอนโด แคมปัส ไฮด์อเวย์":
        "dcondo Campus Hideaway (ดีคอนโด แคมปัส ไฮด์อเวย์)",
    "ดีคอนโด แคมปัส โดม - รังสิต":
        "dcondo campus Dome - Rangsit (ดีคอนโด แคมปัส โดม - รังสิต)",
    "ดีคอนโด แคมปัส รังสิต 2":
        "D condo campus rangsit 2 (ดีคอนโด แคมปัส รังสิต 2)",

    # =====================================================
    # OTHER CONDO
    # =====================================================

    "แอททิจูด บียู":
        "Attitude BU (แอททิจูด บียู)",
    "เทอร์ร่า เรสซิเดนซ์":
        "TERRA RESIDENCE (เทอร์ร่า เรสซิเดนซ์)",
    "คอมมอน ทียู":
        "Common TU (คอมมอน ทียู)",
    "จิณณ์ เวลบีอิ้ง เคาน์ตี้":
        "Jin Wellbeing County (จิณณ์ เวลบีอิ้ง เคาน์ตี้)",
    "บี คอนโด พหลโยธิน":
        "Be Condo Phaholyothin (บี คอนโด พหลโยธิน)",
    "เดอะ คิทท์ คลองหลวง":
        "The Kith Khlong Luang (เดอะ คิทท์ คลองหลวง)",
    "ซูม คอนโด 49":
        "Zoom Condo 49 (ซูม คอนโด 49)",
    "ฟ้าโดม":
        "Fah Dome (ฟ้าโดม)",
    "แกรนด์ โมเดิร์น คอนโด":
        "Grand Modern Condo (แกรนด์ โมเดิร์น คอนโด)",
    "บ้านเอื้ออาทรพหลโยธิน ซอยคุณพระ":
        "Baan Aeu Arthorn PhahonYothin Soi Khun Phra (บ้านเอื้ออาทรพหลโยธิน ซอยคุณพระ)",

    # =====================================================
    # PLOEN PLOEN
    # =====================================================

    "เพลิน เพลิน เจน วายด์ รังสิต - คลองหลวง":
        "Ploen Ploen GEN WIDE Rangsit - Khlong Luang (เพลิน เพลิน เจน วายด์ รังสิต - คลองหลวง)",
    "เพลิน เพลิน เจน วายด์ รังสิต - คลองหลวง 2":
        "Ploen Ploen GEN WIDE Rangsit - Khlong Luang 2 (เพลิน เพลิน เจน วายด์ รังสิต - คลองหลวง 2)",

    # =====================================================
    # BAAN
    # =====================================================

    "บ้านร่วมทางฝัน 3":
        "Baan Ruam Tang Fun 3 (บ้านร่วมทางฝัน 3)",
}

In [ ]:
df["project_name"] = df["project_name"].map(project_map)
df.head()

,project_name,url,price,room_type,room_size_sqm,floor,building,latitude,longitude,room_facilities,project_facilities,image_1,image_2,image_3,image_4
0,dcondo Shine Rangsit (ดีคอนโด ชายน์ รังสิต),https://propertyhub.in.th/listings/26-ตร-ม-11-...,11000.0,ONE_BED_ROOM,26.00,8,None,14.062188,100.608401,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ , เครื่องทำน้ำ...","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...
1,dcondo Campus Resort Rangsit (ดีคอนโด แคมปัส ร...,https://propertyhub.in.th/listings/rb23922-ให้...,8000.0,STUDIO,29.00,2,A,14.063015,100.607360,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...
2,dcondo hype rangsit (ดีคอนโด ไฮป์ รังสิต),https://propertyhub.in.th/listings/ให้เช่า-d-c...,9000.0,ONE_BED_ROOM,27.52,3,None,14.041953,100.616681,เฟอร์นิเจอร์,"ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...
3,Kave Wonderland (เคฟ วันเดอร์แลนด์),https://propertyhub.in.th/listings/เช่าคอนโด-เ...,13000.0,ONE_BED_ROOM,25.00,3,None,14.065818,100.611212,NaN,"ลิฟท์ , ที่จอดรถจักรยานยนต์ , การรักษาความปลอด...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...
4,dcondo vivid Rangsit (ดีคอนโด วิวิด รังสิต),https://propertyhub.in.th/listings/ให้เช่า-dco...,11000.0,ONE_BED_ROOM,1.00,4,A,14.041670,100.616693,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ , เครื่องทำน้ำ...","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...


In [ ]:
# ดู unique room_type
df["room_type"].unique()

array(['ONE_BED_ROOM', 'STUDIO', 'TWO_BED_ROOM', 'DUPLEX_ONE_BED'],
      dtype=object)

In [ ]:
room_type_map = {

    "ONE_BED_ROOM": "1 ห้องนอน",
    "STUDIO": "สตูดิโอ",
    "TWO_BED_ROOM": "2 ห้องนอน",
    "DUPLEX_ONE_BED": "ดูเพล็กซ์ 1 ห้องนอน",
}

In [ ]:
df["room_type"] = (df["room_type"].map(room_type_map))
df.head()

,project_name,url,price,room_type,room_size_sqm,floor,building,latitude,longitude,room_facilities,project_facilities,image_1,image_2,image_3,image_4
0,dcondo Shine Rangsit (ดีคอนโด ชายน์ รังสิต),https://propertyhub.in.th/listings/26-ตร-ม-11-...,11000.0,1 ห้องนอน,26.00,8,None,14.062188,100.608401,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ , เครื่องทำน้ำ...","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...
1,dcondo Campus Resort Rangsit (ดีคอนโด แคมปัส ร...,https://propertyhub.in.th/listings/rb23922-ให้...,8000.0,สตูดิโอ,29.00,2,A,14.063015,100.607360,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...,https://bcdn.propertyhub.in.th/pictures/202603...
2,dcondo hype rangsit (ดีคอนโด ไฮป์ รังสิต),https://propertyhub.in.th/listings/ให้เช่า-d-c...,9000.0,1 ห้องนอน,27.52,3,None,14.041953,100.616681,เฟอร์นิเจอร์,"ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...
3,Kave Wonderland (เคฟ วันเดอร์แลนด์),https://propertyhub.in.th/listings/เช่าคอนโด-เ...,13000.0,1 ห้องนอน,25.00,3,None,14.065818,100.611212,NaN,"ลิฟท์ , ที่จอดรถจักรยานยนต์ , การรักษาความปลอด...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...
4,dcondo vivid Rangsit (ดีคอนโด วิวิด รังสิต),https://propertyhub.in.th/listings/ให้เช่า-dco...,11000.0,1 ห้องนอน,1.00,4,A,14.041670,100.616693,"เฟอร์นิเจอร์ , เครื่องปรับอากาศ , เครื่องทำน้ำ...","ลิฟท์ , ที่จอดรถ , ที่จอดรถจักรยานยนต์ , การรั...",https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...,https://bcdn.propertyhub.in.th/pictures/202605...


In [ ]:
# =========================================================
# CLEAN BUILDING
# =========================================================

def clean_building(text):

    # -----------------------------------------------------
    # NONE
    # -----------------------------------------------------

    if text is None:
        return ""

    # -----------------------------------------------------
    # LIST / ARRAY
    # -----------------------------------------------------

    if isinstance(text, (list, tuple)):

        if len(text) == 0:
            return ""

        text = text[0]

    # -----------------------------------------------------
    # TO STRING
    # -----------------------------------------------------

    text = str(text).strip()

    # -----------------------------------------------------
    # EMPTY
    # -----------------------------------------------------

    if text == "":
        return ""

    # -----------------------------------------------------
    # LOWER
    # -----------------------------------------------------

    lower_text = text.lower()

    # -----------------------------------------------------
    # BAD VALUES
    # -----------------------------------------------------

    bad_keywords = [
        "{'thumbnailUrl': '/pictures/202211/20221117/QJx1qqNNo1k7cwnEuuph/eb260bec.jpg', 'pictureUrl': '/pictures/202211/20221117/QJx1qqNNo1k7cwnEuuph/be7210af.jpg'}",
        "{'thumbnailUrl': '/pictures/202502/20250211/frRVd7ie77o1VVkrNKa4/eb260bec.jpg', 'pictureUrl': '/pictures/202502/20250211/frRVd7ie77o1VVkrNKa4/be7210af.jpg'} ",
        "{'thumbnailUrl': '/pictures/202204/20220418/KDKpR6FgXksiwZSPkGx2/eb260bec.jpg', 'pictureUrl': '/pictures/202204/20220418/KDKpR6FgXksiwZSPkGx2/be7210af.jpg'}",
    ]

    if lower_text in bad_keywords:
        return ""

    # -----------------------------------------------------
    # ลบคำว่า ตึก / อาคาร
    # -----------------------------------------------------

    text = re.sub(
        r"(ตึก|อาคาร)\s*",
        "",
        text,
        flags=re.IGNORECASE
    )

    text = text.strip()

    # -----------------------------------------------------
    # เฟส 1 B -> B เฟส 1
    # -----------------------------------------------------

    phase_match = re.search(
        r"เฟส\s*(\d+).*?([A-Z])",
        text,
        flags=re.IGNORECASE
    )

    if phase_match:

        phase_num = phase_match.group(1)
        building = phase_match.group(2).upper()

        return f"{building} เฟส {phase_num}"

    # -----------------------------------------------------
    # SINGLE BUILDING
    # -----------------------------------------------------

    single_building = re.fullmatch(
        r"[A-Z]",
        text,
        flags=re.IGNORECASE
    )

    if single_building:
        return text.upper()

    # -----------------------------------------------------
    # NUMBER 1-10
    # -----------------------------------------------------

    single_number = re.fullmatch(
        r"(10|[1-9])",
        text
    )

    if single_number:
        return text

    # -----------------------------------------------------
    # B or C
    # -----------------------------------------------------

    multi_building = re.fullmatch(
        r"[A-Z]\s*(or|/|&)\s*[A-Z]",
        text,
        flags=re.IGNORECASE
    )

    if multi_building:
        return text.upper()

    # -----------------------------------------------------
    # CLEAN SPACE
    # -----------------------------------------------------

    text = re.sub(r"\s+", " ", text)

    return text.strip()

# =========================================================
# APPLY
# =========================================================

df["building"] = df["building"].apply(clean_building)

# =========================================================
# CHECK
# =========================================================

print(df["building"].value_counts(dropna=False).head(50))

building
                                                                                                                                                                391
B                                                                                                                                                               157
A                                                                                                                                                               138
C                                                                                                                                                               103
D                                                                                                                                                                73
E                                                                                                                                                                33
F      

In [ ]:
# =========================================================
# FIX BUILDING COLUMN
# =========================================================

def fix_building(value):

    # -----------------------------------------------------
    # NULL
    # -----------------------------------------------------

    if pd.isna(value):
        return ""

    text = str(value).strip()

    # -----------------------------------------------------
    # ซี -> C
    # -----------------------------------------------------

    if text == "ซี":
        return "C"

    # -----------------------------------------------------
    # dict image object -> ""
    # -----------------------------------------------------

    if "thumbnailUrl" in text or "pictureUrl" in text:
        return ""

    return text

# =========================================================
# APPLY
# =========================================================

df["building"] = df["building"].apply(fix_building)

# =========================================================
# CHECK
# =========================================================

print(df["building"].value_counts(dropna=False).head(50))

building
           394
B          157
A          138
C          104
D           73
E           33
F            7
1            4
G            4
2            2
เฟส1         1
C เฟส 3      1
B เฟส 1      1
A เฟส 1      1
B เฟส 3      1
B เฟส 2      1
3C           1
D เฟส 1      1
B7           1
B5           1
B OR C       1
ฺB           1
B8           1
9            1
4            1
B3           1
Name: count, dtype: int64


In [ ]:
# =========================================================
# IMAGE COLUMNS
# =========================================================

image_cols = [
    "image_1",
    "image_2",
    "image_3",
    "image_4"
]

# =========================================================
# FILL EMPTY IMAGE
# =========================================================

for col in image_cols:

    df[col] = df[col].fillna("ไม่มีรูปภาพ")

    df[col] = df[col].replace(
        ["", " ", "nan", "None"],
        "ไม่มีรูปภาพ"
    )

# =========================================================
# CHECK
# =========================================================

print(
    df[image_cols]
    .head()
)

                                             image_1  \
0  https://bcdn.propertyhub.in.th/pictures/202603...   
1  https://bcdn.propertyhub.in.th/pictures/202603...   
2  https://bcdn.propertyhub.in.th/pictures/202605...   
3  https://bcdn.propertyhub.in.th/pictures/202605...   
4  https://bcdn.propertyhub.in.th/pictures/202605...   

                                             image_2  \
0  https://bcdn.propertyhub.in.th/pictures/202603...   
1  https://bcdn.propertyhub.in.th/pictures/202603...   
2  https://bcdn.propertyhub.in.th/pictures/202605...   
3  https://bcdn.propertyhub.in.th/pictures/202605...   
4  https://bcdn.propertyhub.in.th/pictures/202605...   

                                             image_3  \
0  https://bcdn.propertyhub.in.th/pictures/202603...   
1  https://bcdn.propertyhub.in.th/pictures/202603...   
2  https://bcdn.propertyhub.in.th/pictures/202605...   
3  https://bcdn.propertyhub.in.th/pictures/202605...   
4  https://bcdn.propertyhub.in.th/pictures/202

In [ ]:
print(df.isna().sum())

project_name           0
url                    0
price                 10
room_type              0
room_size_sqm          0
floor                  1
building               0
latitude               0
longitude              0
room_facilities       45
project_facilities     4
image_1                0
image_2                0
image_3                0
image_4                0
dtype: int64


In [ ]:
# =========================================================
# CLEAN FLOOR
# =========================================================

def clean_floor(value):

    if pd.isna(value):
        return ""

    text = str(value).strip().upper()

    # KEEP FLOOR RANGE
    if re.fullmatch(r"\d+\s*-\s*\d+", text):
        return text.replace(" ", "")

    # KEEP 12A / 8B
    if re.fullmatch(r"\d+[A-Z]", text):
        return text

    # ลบอักขระแปลก ๆ
    text = re.sub(r"[^\dA-Z]", "", text)

    # EMPTY
    if text == "":
        return ""

    # KEEP NUMBER
    if text.isdigit():
        return text

    # KEEP AGAIN AFTER CLEAN
    if re.fullmatch(r"\d+[A-Z]", text):
        return text

    return ""

df["floor"] = df["floor"].apply(clean_floor)

print(
    df["floor"]
    .value_counts(dropna=False)
    .sort_index()
)

floor
         12
1        23
1-10      1
1-5       3
1-8       2
10        4
11        2
12        7
12A       1
14        4
15        1
16        1
18        6
19        4
2       118
20        2
21        3
22        2
23       13
24       18
25        1
26        3
27       10
28        6
29        4
3       139
30        6
31        3
32        1
4        91
5       101
5-10      3
6       120
6-10      2
7        88
8       120
9         7
Name: count, dtype: int64


In [ ]:
# ROOM FACILITIES
df["room_facilities"] = (
    df["room_facilities"]
    .fillna("")
    .replace(
        ["", " ", "nan", "None"],
        "ไม่มีสิ่งอำนวยความสะดวกภายในห้อง"
    )
)

# PROJECT FACILITIES
df["project_facilities"] = (
    df["project_facilities"]
    .fillna("")
    .replace(
        ["", " ", "nan", "None"],
        "ไม่มีสิ่งอำนวยความสะดวกภายในโครงการ"
    )
)

print(df.isna().sum())

project_name           0
url                    0
price                 10
room_type              0
room_size_sqm          0
floor                  0
building               0
latitude               0
longitude              0
room_facilities        0
project_facilities     0
image_1                0
image_2                0
image_3                0
image_4                0
dtype: int64


In [ ]:
# price
df["price"] = (
    df["price"]
    .fillna("")
    .replace(
        ["", " ", "nan", "None"],
        "ไม่ระบุราคา"
    )
)

print(df.isna().sum())

project_name          0
url                   0
price                 0
room_type             0
room_size_sqm         0
floor                 0
building              0
latitude              0
longitude             0
room_facilities       0
project_facilities    0
image_1               0
image_2               0
image_3               0
image_4               0
dtype: int64


# 🌟 **save file**

In [ ]:
csv_name = "bangkok_university_dorms.csv"

df.to_csv(
    csv_name,
    index=False,
    encoding="utf-8-sig"
)

print("\nDONE")
print("TOTAL =", len(df))
print("CSV =", csv_name)



DONE
TOTAL = 932
CSV = bangkok_university_dorms.csv
